<center><img src="https://github.com/DACSS-PreProcessing/Week_1_main/blob/main/pics/LogoSimple.png?raw=true" width="700"></center>

# Data Formatting in Python

Let me collect some data from the [web](https://en.wikipedia.org/wiki/List_of_freedom_indices):

In [1]:
import pandas as pd

wikiLink="https://en.wikipedia.org/wiki/List_of_freedom_indices" 
freedomDFs=pd.read_html(wikiLink, flavor='bs4',attrs={'class':'sortable'})
len(freedomDFs)

2

The one we need is the first one:

In [2]:
freedomDFs[0].head()

,Country,Freedom in the World 2024[16],Score change since 2023,Index of Economic Freedom 2024[17],Score,Press Freedom Index 2023[3],Score.1,Democracy Index 2023[18],Score.2
0,Finland,100.0,0,mostly free,76.3,good,87.94,full democracy,9.30
1,New Zealand,99.0,0,mostly free,77.8,satisfactory,84.23,full democracy,9.61
2,Sweden,99.0,-1,mostly free,77.5,good,88.15,full democracy,9.39
3,Norway,98.0,-2,mostly free,77.5,good,95.18,full democracy,9.81
4,Canada,97.0,-1,mostly free,72.4,satisfactory,83.53,full democracy,8.69


Let's keep it:

In [3]:
freedom=freedomDFs[0].copy()
freedom.head()

,Country,Freedom in the World 2024[16],Score change since 2023,Index of Economic Freedom 2024[17],Score,Press Freedom Index 2023[3],Score.1,Democracy Index 2023[18],Score.2
0,Finland,100.0,0,mostly free,76.3,good,87.94,full democracy,9.30
1,New Zealand,99.0,0,mostly free,77.8,satisfactory,84.23,full democracy,9.61
2,Sweden,99.0,-1,mostly free,77.5,good,88.15,full democracy,9.39
3,Norway,98.0,-2,mostly free,77.5,good,95.18,full democracy,9.81
4,Canada,97.0,-1,mostly free,72.4,satisfactory,83.53,full democracy,8.69


## The cleaning process

### cleaning the headers

We need to clean before formatting.

In [4]:
# check headers
freedom.columns

Index(['Country', 'Freedom in the World 2024[16]', 'Score change since 2023',
       'Index of Economic Freedom 2024[17]', 'Score',
       'Press Freedom Index 2023[3]', 'Score.1', 'Democracy Index 2023[18]',
       'Score.2'],
      dtype='object')

The plan here is:

1. No footnotes
2. No year
3. No trailing/leading spaces
4. Lower case for all names
5. No unwanted columns (third column)
6. Try the simplest names that keep meaning of the column
7. Give proper names that benefit similarity and difference

Let's start:

**1. No footnotes**

In [6]:
patternFootnotes=r'\[.*\]'
freedom.columns=freedom.columns.str.replace(patternFootnotes,"",regex=True)
freedom.columns

Index(['Country', 'Freedom in the World 2024', 'Score change since 2023',
       'Index of Economic Freedom 2024', 'Score', 'Press Freedom Index 2023',
       'Score.1', 'Democracy Index 2023', 'Score.2'],
      dtype='object')

**2. No year**

In [7]:
patternYear=r'\d{4}'

freedom.columns=freedom.columns.str.replace(patternYear,"",regex=True)
freedom.columns

Index(['Country', 'Freedom in the World ', 'Score change since ',
       'Index of Economic Freedom ', 'Score', 'Press Freedom Index ',
       'Score.1', 'Democracy Index ', 'Score.2'],
      dtype='object')

**3. No trailing/leading spaces**

In [8]:
freedom.columns=freedom.columns.str.strip()
freedom.columns

Index(['Country', 'Freedom in the World', 'Score change since',
       'Index of Economic Freedom', 'Score', 'Press Freedom Index', 'Score.1',
       'Democracy Index', 'Score.2'],
      dtype='object')

**4. Lower case for all names**

In [9]:
freedom.columns=freedom.columns.str.lower()
freedom.columns

Index(['country', 'freedom in the world', 'score change since',
       'index of economic freedom', 'score', 'press freedom index', 'score.1',
       'democracy index', 'score.2'],
      dtype='object')

In [10]:
### the last four steps could have been done this way

# pattern_NoYear_NoFootns=r'\[.*\]|\d{4}' # this shortens coding

# freedom.columns=freedom.columns.str.replace(pattern_NoYear_NoFootns,"",regex=True).str.strip().str.lower()

**5. No unwanted columns (third column)**

In [11]:
freedom.drop(freedom.columns[2], axis=1,inplace=True)
freedom.columns

Index(['country', 'freedom in the world', 'index of economic freedom', 'score',
       'press freedom index', 'score.1', 'democracy index', 'score.2'],
      dtype='object')

**6. Try the simplest names that keep meaning of the column**

In [12]:
notAddingMeaning=r"\s|index|of|freedom|in|the"

freedom.columns=freedom.columns.str.replace(notAddingMeaning,"",regex=True)
freedom.columns

Index(['country', 'world', 'economic', 'score', 'press', 'score.1',
       'democracy', 'score.2'],
      dtype='object')

**7. Give proper names that benefit similarity and difference**

There are columns with scores, and other with categories, let's use that in the names:

In [13]:
# preparing changes in a dict
newNamesForscores={bad:better for bad,better in zip(freedom.columns[3::2],freedom.columns[2::2]+'_score')}
newNamesForscores

{'score': 'economic_score',
 'score.1': 'press_score',
 'score.2': 'democracy_score'}

In [14]:
# renaming
freedom.rename(columns=newNamesForscores,inplace=True)
freedom

,country,world,economic,economic_score,press,press_score,democracy,democracy_score
0,Finland,100.0,mostly free,76.3,good,87.94,full democracy,9.30
1,New Zealand,99.0,mostly free,77.8,satisfactory,84.23,full democracy,9.61
2,Sweden,99.0,mostly free,77.5,good,88.15,full democracy,9.39
3,Norway,98.0,mostly free,77.5,good,95.18,full democracy,9.81
4,Canada,97.0,mostly free,72.4,satisfactory,83.53,full democracy,8.69
...,...,...,...,...,...,...,...,...
192,North Korea,3.0,repressed,2.9,very serious,21.72,authoritarian,1.08
193,Turkmenistan,2.0,repressed,46.3,very serious,25.82,authoritarian,1.66
194,South Sudan,1.0,NaN,—,difficult,50.62,NaN,—
195,Syria,1.0,NaN,—,very serious,27.22,authoritarian,1.43


Since we have Freedom in the World as a _score_, let's rename it:

In [15]:
freedom.rename(columns={'world':'world_score'},inplace=True)

We should give a particular name to categories:

In [16]:
newNamesForCat={old:new for old,new in zip(freedom.columns[2::2],freedom.columns[2::2]+'_level')}

freedom.rename(columns=newNamesForCat,inplace=True)
freedom

,country,world_score,economic_level,economic_score,press_level,press_score,democracy_level,democracy_score
0,Finland,100.0,mostly free,76.3,good,87.94,full democracy,9.30
1,New Zealand,99.0,mostly free,77.8,satisfactory,84.23,full democracy,9.61
2,Sweden,99.0,mostly free,77.5,good,88.15,full democracy,9.39
3,Norway,98.0,mostly free,77.5,good,95.18,full democracy,9.81
4,Canada,97.0,mostly free,72.4,satisfactory,83.53,full democracy,8.69
...,...,...,...,...,...,...,...,...
192,North Korea,3.0,repressed,2.9,very serious,21.72,authoritarian,1.08
193,Turkmenistan,2.0,repressed,46.3,very serious,25.82,authoritarian,1.66
194,South Sudan,1.0,NaN,—,difficult,50.62,NaN,—
195,Syria,1.0,NaN,—,very serious,27.22,authoritarian,1.43


### Cleaning the contents

The plan at this stage is:

1. Do preventive cleaning in columns with text
2. Verify the levels of categorical values have no mistypings
3. Check what is causing that numeric data columns are interpreted in a differen way.

Let's start!

**1. Do preventive cleaning in columns with text**

Here, we get rid of trailing/leading spaces in cells with strings

In [22]:
# which are the non numeric
nonNumericCols=freedom.columns[~freedom.columns.str.endswith('score')]
nonNumericCols

Index(['country', 'economic_level', 'press_level', 'democracy_level'], dtype='object')

In [23]:
# preventive cleaning

freedom.loc[:,nonNumericCols]=freedom.loc[:,nonNumericCols].apply(lambda x:x.str.strip())

**2. Verify the levels of categorical values have no mistypings**

In [25]:
# just an exploration.
catCols=freedom.columns[freedom.columns.str.endswith('level')]
freedom.loc[:,catCols].apply(lambda x: pd.unique(x)).values

array([array(['mostly free', 'free', nan, 'moderately free', 'mostly unfree',
              'repressed'], dtype=object)                                    ,
       array(['good', 'satisfactory', nan, 'problematic', 'difficult',
              'very serious'], dtype=object)                          ,
       array(['full democracy', nan, 'flawed democracy', 'hybrid regime',
              'authoritarian'], dtype=object)                            ],
      dtype=object)

The categories are well written. Even the missing values are well represented.

Let's do the third step; here, we see how the numeric columns are identified:

In [27]:
freedom.loc[:,NumericCols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   world_score      196 non-null    float64
 1   economic_score   197 non-null    object 
 2   press_score      197 non-null    object 
 3   democracy_score  197 non-null    object 
dtypes: float64(1), object(3)
memory usage: 6.3+ KB


**3. Check what is causing that numeric data columns are interpreted in a differen way.**

Three columns that are numeric are not read as numeric (_float_). Let's solve that.

In [29]:
NumericColsBad=NumericCols[1:]

In [30]:
# show me the cells that do not resemble numeric as xx.xx:
numericFormat=r'^\d+.*\d*$'

NumericCols=freedom.columns[freedom.columns.str.endswith('score')]
freedom.loc[:,NumericColsBad].apply(lambda x: pd.unique(x[~x.str.contains(numericFormat)]))

,economic_score,press_score,democracy_score
0,—,—,—


I am showing the inappropriate symbols  present in each column. 

Let me show you an example in a case where you have more wrong values:

In [31]:
# instead of
freedom.loc[0,'economic_score']

'76.3'

In [32]:
# this
freedom.loc[0,'economic_score']='X'

You would get this:

In [34]:
freedom.loc[:,NumericColsBad].apply(lambda x: pd.unique(x[~x.str.contains(r'^\d+.*\d*$')]))

economic_score     [X, —]
press_score           [—]
democracy_score       [—]
dtype: object

Then, the general way to recover this values could be:

In [36]:
import numpy as np

set(np.concatenate(freedom.loc[:,NumericColsBad].apply(lambda x: pd.unique(x[~x.str.contains(r'^\d+.*\d*$')])).values))

{'X', '—'}

Let me undo the change:

In [37]:
freedom.loc[0,'economic_score']='76.3'

Let's continue.

Those characters are used to show missing values. We need to get rid of them in a **proper way**:

* Identify the character(s):

In [38]:
badSymbols=set(freedom.loc[:,NumericColsBad].apply(lambda x: pd.unique(x[~x.str.contains(r'^\d+.*\d*$')])).values.flatten())
badSymbols

{'—'}

* Replace those by missing values:

In [41]:
freedom.loc[:,NumericColsBad]=freedom.loc[:,NumericColsBad].replace(badSymbols,None)

* If needed, keep complete data:

In [42]:
freedom.dropna(how='any', ignore_index=True, inplace=True) 

# Time to format the data

When formatting data we pay attention to the data contents.
The data contents are clean, but not yet formatted:

In [43]:
freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          159 non-null    object 
 1   world_score      159 non-null    float64
 2   economic_level   159 non-null    object 
 3   economic_score   159 non-null    object 
 4   press_level      159 non-null    object 
 5   press_score      159 non-null    object 
 6   democracy_level  159 non-null    object 
 7   democracy_score  159 non-null    object 
dtypes: float64(1), object(7)
memory usage: 10.1+ KB


## Formatting numerical values


Let's format the numeric data using **pd.to_numeric**. 

Remember this:

In [44]:
# gives an error
# pd.to_numeric(pd.Series(['1','2','$3']),errors='raise')

In [45]:
# bad values to missing
pd.to_numeric(pd.Series(['1','2','$3']),errors='coerce')

0    1.0
1    2.0
2    NaN
dtype: float64

In [46]:
# bad values stay
pd.to_numeric(pd.Series(['1','2','$3']),errors='ignore')

0     1
1     2
2    $3
dtype: object

You should not coerce, because you can be deleting good values which are poorly written. Then,

In [48]:
freedom[NumericColsBad]=freedom.loc[:,NumericColsBad].apply(lambda x: pd.to_numeric(x,errors='raise'))

freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          159 non-null    object 
 1   world_score      159 non-null    float64
 2   economic_level   159 non-null    object 
 3   economic_score   159 non-null    float64
 4   press_level      159 non-null    object 
 5   press_score      159 non-null    float64
 6   democracy_level  159 non-null    object 
 7   democracy_score  159 non-null    float64
dtypes: float64(4), object(4)
memory usage: 10.1+ KB


Great! - the formatting did not abort.

## Formatting the categories

Our categories are ordinal:

In [51]:
freedom.loc[:,catCols].apply(lambda x: pd.unique(x)).values

array([array(['mostly free', 'free', 'moderately free', 'repressed',
              'mostly unfree'], dtype=object)                       ,
       array(['good', 'satisfactory', 'problematic', 'difficult', 'very serious'],
             dtype=object)                                                        ,
       array(['full democracy', 'flawed democracy', 'hybrid regime',
              'authoritarian'], dtype=object)                       ],
      dtype=object)

The plan is:

1. Use integers instead of text. If labels accross columns are not the same, use same values for max and min.
2. Notice all scores have categories in another column, except 'World_score'. Create his column of categories as integers.
3. Create alternative columns, where you can write labels, but include numbers in the labels.

* **This solves the first case:**

In [52]:
# maps for replacement: 1 the worst / 5 the best
mapper1 = {'repressed':1, 'mostly unfree':2,'moderately free':3, 'mostly free':4, 'free':5}
mapper2 = {'very serious':1, 'difficult':2,'problematic':3,'satisfactory':4,'good':5}
mapper3 = {'authoritarian':1,'hybrid regime':2,'flawed democracy':4, 'full democracy':5}


freedom.economic_level.replace(mapper1,inplace=True)
freedom.press_level.replace(mapper2,inplace=True)
freedom.democracy_level.replace(mapper3,inplace=True)

* **For the second case we can CUT the variable with qcut:**

In [53]:
# from the methodology, the index has 3 levels, we use that:
freedom['world_level']=pd.qcut(freedom.world_score, 3, labels=False)
freedom.world_level.value_counts(sort=False)

world_level
2    51
1    55
0    53
Name: count, dtype: int64

In [54]:
# to standardize
freedom.world_level.replace({0:1,1:3,2:5},inplace=True)

In [55]:
# currently:
freedom.head()

,country,world_score,economic_level,economic_score,press_level,press_score,democracy_level,democracy_score,world_level
0,Finland,100.0,4,76.3,5,87.94,5,9.30,5
1,New Zealand,99.0,4,77.8,4,84.23,5,9.61,5
2,Sweden,99.0,4,77.5,5,88.15,5,9.39,5
3,Norway,98.0,4,77.5,5,95.18,5,9.81,5
4,Canada,97.0,4,72.4,4,83.53,5,8.69,5


In [56]:
freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          159 non-null    object 
 1   world_score      159 non-null    float64
 2   economic_level   159 non-null    int64  
 3   economic_score   159 non-null    float64
 4   press_level      159 non-null    int64  
 5   press_score      159 non-null    float64
 6   democracy_level  159 non-null    int64  
 7   democracy_score  159 non-null    float64
 8   world_level      159 non-null    int64  
dtypes: float64(4), int64(4), object(1)
memory usage: 11.3+ KB


Let me put the last variable in a better location:

In [57]:
# just a trick
freedom=freedom.set_index(['country','world_level']).reset_index(drop=False)
freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          159 non-null    object 
 1   world_level      159 non-null    int64  
 2   world_score      159 non-null    float64
 3   economic_level   159 non-null    int64  
 4   economic_score   159 non-null    float64
 5   press_level      159 non-null    int64  
 6   press_score      159 non-null    float64
 7   democracy_level  159 non-null    int64  
 8   democracy_score  159 non-null    float64
dtypes: float64(4), int64(4), object(1)
memory usage: 11.3+ KB


* **This is the last step:**

In [58]:
# new column names
simpleNames=[x[0] for x in freedom.columns[freedom.columns.str.endswith('level')].str.split('_')]
simpleNames

['world', 'economic', 'press', 'democracy']

In [59]:
newColNames=[n+'_label' for n in simpleNames]
newColNames

['world_label', 'economic_label', 'press_label', 'democracy_label']

In [60]:
# copy the previous values
freedom[newColNames]=freedom.iloc[:,1::2]
freedom.head()

,country,world_level,world_score,economic_level,economic_score,press_level,press_score,democracy_level,democracy_score,world_label,economic_label,press_label,democracy_label
0,Finland,5,100.0,4,76.3,5,87.94,5,9.30,5,4,5,5
1,New Zealand,5,99.0,4,77.8,4,84.23,5,9.61,5,4,4,5
2,Sweden,5,99.0,4,77.5,5,88.15,5,9.39,5,4,5,5
3,Norway,5,98.0,4,77.5,5,95.18,5,9.81,5,4,5,5
4,Canada,5,97.0,4,72.4,4,83.53,5,8.69,5,4,4,5


In [61]:
# create the data type info
from pandas.api.types import CategoricalDtype
myOrdinal = CategoricalDtype(categories=[1,2,3,4,5], ordered=True)

#one column
freedom.loc[:,"world_label"].astype(myOrdinal)

0      5
1      5
2      5
3      5
4      5
      ..
154    1
155    1
156    1
157    1
158    1
Name: world_label, Length: 159, dtype: category
Categories (5, int64): [1 < 2 < 3 < 4 < 5]

In [62]:
# several columns
freedom.loc[:,"world_label":]=freedom.loc[:,"world_label":].astype(myOrdinal)
freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   country          159 non-null    object  
 1   world_level      159 non-null    int64   
 2   world_score      159 non-null    float64 
 3   economic_level   159 non-null    int64   
 4   economic_score   159 non-null    float64 
 5   press_level      159 non-null    int64   
 6   press_score      159 non-null    float64 
 7   democracy_level  159 non-null    int64   
 8   democracy_score  159 non-null    float64 
 9   world_label      159 non-null    category
 10  economic_label   159 non-null    category
 11  press_label      159 non-null    category
 12  democracy_label  159 non-null    category
dtypes: category(4), float64(4), int64(4), object(1)
memory usage: 12.8+ KB


Finally, rename the labels:

In [63]:
# rename the levels

ordinalLevels={1:'1_veryLow',2:'2_low',3:'3_medium',4:'4_good',5:'5_veryGood'}

renameLevels= lambda x:x.cat.rename_categories(ordinalLevels)

freedom.loc[:,"world_label":]=freedom.loc[:,"world_label":].apply(renameLevels)

The final result:

In [64]:
freedom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   country          159 non-null    object  
 1   world_level      159 non-null    int64   
 2   world_score      159 non-null    float64 
 3   economic_level   159 non-null    int64   
 4   economic_score   159 non-null    float64 
 5   press_level      159 non-null    int64   
 6   press_score      159 non-null    float64 
 7   democracy_level  159 non-null    int64   
 8   democracy_score  159 non-null    float64 
 9   world_label      159 non-null    category
 10  economic_label   159 non-null    category
 11  press_label      159 non-null    category
 12  democracy_label  159 non-null    category
dtypes: category(4), float64(4), int64(4), object(1)
memory usage: 12.8+ KB


In [65]:
freedom.head()

,country,world_level,world_score,economic_level,economic_score,press_level,press_score,democracy_level,democracy_score,world_label,economic_label,press_label,democracy_label
0,Finland,5,100.0,4,76.3,5,87.94,5,9.30,5_veryGood,4_good,5_veryGood,5_veryGood
1,New Zealand,5,99.0,4,77.8,4,84.23,5,9.61,5_veryGood,4_good,4_good,5_veryGood
2,Sweden,5,99.0,4,77.5,5,88.15,5,9.39,5_veryGood,4_good,5_veryGood,5_veryGood
3,Norway,5,98.0,4,77.5,5,95.18,5,9.81,5_veryGood,4_good,5_veryGood,5_veryGood
4,Canada,5,97.0,4,72.4,4,83.53,5,8.69,5_veryGood,4_good,4_good,5_veryGood


We did not have dates in this data file, so let me check another case.

## Formatting Dates

Let me get the data about [Fire 9-1-1  calls  from Seattle](https://dev.socrata.com/foundry/data.seattle.gov/kzjm-xkqj) we saw in the first week:

In [66]:
import requests

# where is it online?
url = "https://data.seattle.gov/resource/kzjm-xkqj.json"

# Go for the data:
response = requests.get(url)

# If we got the data:
if response.status_code == 200:
    data911 = response.json()

This file gives you as a result a json: 

In [ ]:
# data911

You can turn that json into a dataframe easily:

In [67]:
import pandas as pd
data911DF=pd.DataFrame(data911)

# see
data911DF.head()

,address,type,datetime,latitude,longitude,report_location,incident_number,:@computed_region_ru88_fbhk,:@computed_region_kuhn_3gp2,:@computed_region_q256_3sug,:@computed_region_2day_rhn5,:@computed_region_cyqu_gs94
0,224 Pontius Ave N,Triaged Incident,2024-10-06T19:34:00.000,47.619935,-122.331699,"{'type': 'Point', 'coordinates': [-122.331699,...",F240138532,56,10,18390,NaN,NaN
1,15th Ave Ne / Ne 43rd St,Fast Back Up,2024-10-06T19:27:00.000,47.659781,-122.312032,"{'type': 'Point', 'coordinates': [-122.312032,...",F240138530,60,38,18383,NaN,NaN
2,5000 University Way Ne,Aid Response,2024-10-06T19:22:00.000,47.664901,-122.313074,"{'type': 'Point', 'coordinates': [-122.313074,...",F240138529,60,47,18383,NaN,NaN
3,421 1st Ave S,Medic Response,2024-10-06T19:22:00.000,47.59901,-122.334189,"{'type': 'Point', 'coordinates': [-122.334189,...",F240138528,49,20,18379,NaN,NaN
4,801 Alaskan Way,Aid Response,2024-10-06T19:20:00.000,47.602422,-122.33696,"{'type': 'Point', 'coordinates': [-122.33696, ...",F240138527,14,19,18379,NaN,NaN


Let me get rid of the last five colums:

In [68]:
data911DF=data911DF.iloc[:,:-5]

Let's check the current data types:

In [69]:
data911DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   address          1000 non-null   object
 1   type             1000 non-null   object
 2   datetime         1000 non-null   object
 3   latitude         999 non-null    object
 4   longitude        999 non-null    object
 5   report_location  999 non-null    object
 6   incident_number  1000 non-null   object
dtypes: object(7)
memory usage: 54.8+ KB


As you see, the datetime is not recognized as date:

In [70]:
data911DF.datetime

0      2024-10-06T19:34:00.000
1      2024-10-06T19:27:00.000
2      2024-10-06T19:22:00.000
3      2024-10-06T19:22:00.000
4      2024-10-06T19:20:00.000
                ...           
995    2024-10-04T08:37:00.000
996    2024-10-04T08:37:00.000
997    2024-10-04T08:33:00.000
998    2024-10-04T08:33:00.000
999    2024-10-04T08:28:00.000
Name: datetime, Length: 1000, dtype: object

In [71]:
# it is a string
data911DF.datetime[0],type(data911DF.datetime[0])

('2024-10-06T19:34:00.000', str)

Pandas has this nice function: **pd.to_datetime**:

In [72]:
# voilá
pd.to_datetime(data911DF.datetime)

0     2024-10-06 19:34:00
1     2024-10-06 19:27:00
2     2024-10-06 19:22:00
3     2024-10-06 19:22:00
4     2024-10-06 19:20:00
              ...        
995   2024-10-04 08:37:00
996   2024-10-04 08:37:00
997   2024-10-04 08:33:00
998   2024-10-04 08:33:00
999   2024-10-04 08:28:00
Name: datetime, Length: 1000, dtype: datetime64[ns]

In [73]:
# then,
data911DF['datetime']=pd.to_datetime(data911DF.datetime)

# types?
data911DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   address          1000 non-null   object        
 1   type             1000 non-null   object        
 2   datetime         1000 non-null   datetime64[ns]
 3   latitude         999 non-null    object        
 4   longitude        999 non-null    object        
 5   report_location  999 non-null    object        
 6   incident_number  1000 non-null   object        
dtypes: datetime64[ns](1), object(6)
memory usage: 54.8+ KB


Once you have this column as a datetime, you can do something like this:

In [74]:
data911DF['date']=data911DF.datetime.dt.date
data911DF['year']=data911DF.datetime.dt.year
data911DF['month']=data911DF.datetime.dt.month_name()
data911DF['weekday']=data911DF.datetime.dt.day_name()
data911DF['hour']=data911DF.datetime.dt.hour

In [75]:
data911DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   address          1000 non-null   object        
 1   type             1000 non-null   object        
 2   datetime         1000 non-null   datetime64[ns]
 3   latitude         999 non-null    object        
 4   longitude        999 non-null    object        
 5   report_location  999 non-null    object        
 6   incident_number  1000 non-null   object        
 7   date             1000 non-null   object        
 8   year             1000 non-null   int32         
 9   month            1000 non-null   object        
 10  weekday          1000 non-null   object        
 11  hour             1000 non-null   int32         
dtypes: datetime64[ns](1), int32(2), object(9)
memory usage: 86.1+ KB


In [76]:
# see
data911DF.head()

,address,type,datetime,latitude,longitude,report_location,incident_number,date,year,month,weekday,hour
0,224 Pontius Ave N,Triaged Incident,2024-10-06 19:34:00,47.619935,-122.331699,"{'type': 'Point', 'coordinates': [-122.331699,...",F240138532,2024-10-06,2024,October,Sunday,19
1,15th Ave Ne / Ne 43rd St,Fast Back Up,2024-10-06 19:27:00,47.659781,-122.312032,"{'type': 'Point', 'coordinates': [-122.312032,...",F240138530,2024-10-06,2024,October,Sunday,19
2,5000 University Way Ne,Aid Response,2024-10-06 19:22:00,47.664901,-122.313074,"{'type': 'Point', 'coordinates': [-122.313074,...",F240138529,2024-10-06,2024,October,Sunday,19
3,421 1st Ave S,Medic Response,2024-10-06 19:22:00,47.59901,-122.334189,"{'type': 'Point', 'coordinates': [-122.334189,...",F240138528,2024-10-06,2024,October,Sunday,19
4,801 Alaskan Way,Aid Response,2024-10-06 19:20:00,47.602422,-122.33696,"{'type': 'Point', 'coordinates': [-122.33696, ...",F240138527,2024-10-06,2024,October,Sunday,19


The **date** column can also be converted:

In [77]:
data911DF['date']=pd.to_datetime(data911DF.date,format='%Y-%m-%d')

# changed?
data911DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   address          1000 non-null   object        
 1   type             1000 non-null   object        
 2   datetime         1000 non-null   datetime64[ns]
 3   latitude         999 non-null    object        
 4   longitude        999 non-null    object        
 5   report_location  999 non-null    object        
 6   incident_number  1000 non-null   object        
 7   date             1000 non-null   datetime64[ns]
 8   year             1000 non-null   int32         
 9   month            1000 non-null   object        
 10  weekday          1000 non-null   object        
 11  hour             1000 non-null   int32         
dtypes: datetime64[ns](2), int32(2), object(8)
memory usage: 86.1+ KB


It is good you are aware of the date format language:
* %Y (4-digit year), %y (2-digit year)
* %m (2-digit month), %b (short name of month) %B full name of month
* %d (1-digit month or 2-digit month)

See these examples:

In [78]:
dates=['20240201','20240102']
pd.to_datetime(pd.Series(dates),format='%Y%m%d')

0   2024-02-01
1   2024-01-02
dtype: datetime64[ns]

In [79]:
dates=['2024021','2024012']
pd.to_datetime(pd.Series(dates),format='%Y%m%d')

0   2024-02-01
1   2024-01-02
dtype: datetime64[ns]

In [80]:
dates=['2024/12/10','2024/10/12']
pd.to_datetime(pd.Series(dates),format='%Y/%m/%d')

0   2024-12-10
1   2024-10-12
dtype: datetime64[ns]

In [81]:
dates=['12nov2023','11dec2023']
pd.to_datetime(pd.Series(dates),format='%d%b%Y')

0   2023-11-12
1   2023-12-11
dtype: datetime64[ns]

In [82]:
dates=['NOVEMBER122023','DECEMBER112023']
pd.to_datetime(pd.Series(dates),format='%B%d%Y')

0   2023-11-12
1   2023-12-11
dtype: datetime64[ns]

In [83]:
dates=['NOVEMBER 12,2023','DECEMBER 11,2023']
pd.to_datetime(pd.Series(dates),format='%B %d,%Y')

0   2023-11-12
1   2023-12-11
dtype: datetime64[ns]

### Saving

You should save the formatted data in a way that all those key changes are preserved. Do not use CSV in this stage.

In [84]:
import os

folder = "dataFormatted_py"

# Check whether the specified path exists or not
isExist = os.path.exists(folder)

if not isExist:
   # Create a new directory because it does not exist
   os.makedirs(path)
   freedom.to_pickle(os.path.join(folder, 'freedom.pkl'))
   data911DF.to_pickle(os.path.join(folder, 'data911DF.pkl'))

else:
    freedom.to_pickle(os.path.join(folder, 'freedom.pkl'))
    data911DF.to_pickle(os.path.join(folder, 'data911DF.pkl'))